# Hospital Unified NLP Model — Google Colab & Kaggle GPU Training (100% From Scratch)

Trains a **3-seed `PyTorchCharCNNBiLSTMCRF`** with **Exact Partial-Label Marginal CRF** (`pretrained_weights_used = False`) on NVIDIA CUDA GPU (Google Colab T4 or Kaggle T4/P100) jointly across:
- `SYNTH-MED-V3.1` (EN + FR, 14 entity types)
- `MACCROBAT2020` (EN real case reports, `CC-BY-4.0`)
- `PxCorpus` (FR real clinical prescriptions, `CC-BY-4.0`)
- `E3C Layer 1` (EN + FR clinical narratives, `CC-BY-NC-4.0` — set `PERMISSIVE_ONLY = True` below for commercial-only hospital builds)
- `BC5CDR` (EN Disease + Chemical, `Public-Domain`)
- `NCBI Disease` (EN Disease, `Public-Domain`)

> **Google Colab setup:** Click **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.
> **Privacy rule:** Public / open-license / synthetic data only. Never upload real hospital PHI/PII.

In [ ]:
import os, subprocess, torch
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
print("Environment:", "Kaggle" if WORK_DIR == "/kaggle/working" else "Google Colab / Linux", "| WORK_DIR:", WORK_DIR)
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! In Colab go to Runtime -> Change runtime type -> T4 GPU.")
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# 1. Clone repo & install lightweight project deps (keeping preinstalled CUDA PyTorch)
import os, subprocess
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_URL = "https://github.com/Mtg221/medical_ai.git"
REPO_DIR = os.path.join(WORK_DIR, "medical_ai")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "httpx", "pyyaml", "pydantic", "sklearn-crfsuite", "scikit-learn", "pillow", "fastapi", "uvicorn", "argon2-cffi", "pypdf", "pdfplumber", "pypdfium2", "python-docx", "reportlab", "pytesseract", "python-multipart"], check=True)
subprocess.run(["pip", "install", "-q", "--no-deps", "-e", "."], check=True)
print("Ready in", os.getcwd())

In [ ]:
# 2. Build SYNTH-MED-V3.1 + realistic_llm_80 + download open clinical & public-domain biomedical corpora
import subprocess

# Set True for strictly commercial-permissive corpora (omits CC-BY-NC E3C Layer 1)
PERMISSIVE_ONLY = False

subprocess.run(["medextract", "data", "build", "--config", "configs/data/v3_1.yaml"], check=True)
subprocess.run(["python", "scripts/build_llm_realistic_80.py"], check=True)

dl_cmd = [
    "python", "scripts/download_open_clinical_corpora.py",
    "--corpora", "maccrobat,pxcorpus,e3c,bc5cdr,ncbi_disease",
]
if PERMISSIVE_ONLY:
    dl_cmd.append("--permissive-only")
subprocess.run(dl_cmd, check=True)

# Train from-scratch E3C Layer 3 self-supervised EN/FR SVD-PPMI + semantic cluster embeddings
subprocess.run(["python", "scripts/train_e3c_layer3_embeddings.py"], check=True)

In [ ]:
# 3. Train 3-seed Hospital Char-CNN-BiLSTM-PartialCRF on CUDA GPU & benchmark
import json, subprocess
from pathlib import Path

subprocess.run([
    "python", "scripts/train_hospital_models.py",
    "--seeds", "1,2,3",
    "--epochs", "20",
    "--batch-size", "32",
    "--lr", "0.003",
    "--word-emb-dim", "64",
    "--char-emb-dim", "24",
    "--char-filters", "24",
    "--cluster-emb-dim", "16",
    "--hidden-dim", "128",
    "--num-layers", "2",
    "--dropout", "0.25",
    "--ocr-augment-ratio", "0.15",
    "--device", "auto",
    "--clinical-eval-only",
    "--out-models-dir", "models",
    "--out-report", "docs/results/hospital_partial_crf_gpu_benchmark.json",
], check=True)

report = json.loads(Path("docs/results/hospital_partial_crf_gpu_benchmark.json").read_text(encoding="utf-8"))
for sp, vals in report["splits"].items():
    t = vals["torch_partial_crf"]
    e = vals["hospital_ensemble_torch_plus_crf"]
    print(f"{sp:20s} | 3-seed torch Strict={t['strict_f1']:.4f} (Len={t['lenient_f1']:.4f}) | Ensemble Strict={e['strict_f1']:.4f} (Len={e['lenient_f1']:.4f})")

In [ ]:
# 4. Package trained CPU-loadable checkpoints + SHA-256 manifests & trigger download on Colab
import os, zipfile
from pathlib import Path

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
zip_path = Path(WORK_DIR) / "hospital_nlp_models_gpu.zip"
targets = [
    Path("models/e3c_self_supervised_en_fr.npz"),
    Path("docs/results/hospital_partial_crf_gpu_benchmark.json"),
    *sorted(Path("models").glob("hospital_char_cnn_bilstm_crf_seed*/*")),
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in targets:
        if p.exists() and p.is_file():
            zf.write(p, arcname=str(p))
            print("Added:", p)
print(f"Created {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    print("Download", zip_path, "from the Files / Output sidebar and unzip into your local repo root.")